# 01 · Exploración del dataset — `dataset_modelo.parquet`

**Etapa 4** — paso 1 del flujo: número de registros, columnas, distribución de clases,
valores faltantes y correlaciones.

Usa `polars` en modo *lazy/streaming* para poder trabajar cómodo aunque el archivo pese
~1.83 GB, y **auto-detecta el esquema real** contra lo que produce `preparar_dataset_modelo.py`
de Allan (avisa si algo no coincide, en vez de fallar en silencio).

### ⚠️ RAM local vs. Kabré
Con ~52M filas, materializar todo a `pandas` (paso del split) puede pedir 8-12 GB de RAM.
Si tu laptop no da abasto, deja `MODO = "dev"` para trabajar con una muestra pequeña y
depurar rápido; cuando vayas a correr en Kabré, cambia `MODO = "full"` para procesar el
dataset completo. Ver la última celda para instrucciones de envío a Kabré.

**Entrada:** `../datos/dataset_modelo.parquet`
**Salidas:** `../resultados/tablas/01_resumen_nulos.csv`, `02_balance_clases.csv`,
`../resultados/figuras/01_balance_clases.png`, `02_correlaciones.png`,
`../datos/train.parquet`, `val.parquet`, `test.parquet`


In [ ]:
import os, time
import numpy as np
import pandas as pd
import polars as pl
import plotly.express as px
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Datos pesados (dataset, splits, modelos entrenados) -> viven en /data, NO en /home
USUARIO = os.path.basename(os.path.expanduser("~"))
DATOS_BASE_DIR = os.path.join("/data", USUARIO, "deteccion_incendios")
# Resultados livianos (tablas/figuras que van al repo de Git) -> se quedan en /home
BASE_DIR = os.path.join(os.path.expanduser("~"), "deteccion_incendios", "modelado")
RUTA_DATASET = os.path.join(DATOS_BASE_DIR, "data", "dataset_modelo.parquet")
DIR_DATOS    = os.path.join(DATOS_BASE_DIR, "datos")
DIR_FIGURAS  = os.path.join(BASE_DIR, "resultados/figuras")
DIR_TABLAS   = os.path.join(BASE_DIR, "resultados/tablas")
os.makedirs(DIR_DATOS, exist_ok=True)
os.makedirs(DIR_FIGURAS, exist_ok=True)
os.makedirs(DIR_TABLAS, exist_ok=True)

SEMILLA = 42
TARGET_ESPERADO = "es_falsa_alarma"

# --- MODO DE EJECUCION -------------------------------------------------
# "dev"  -> usa una muestra pequena, corre en segundos, ideal para tu laptop.
# "full" -> usa el dataset completo, para correr en Kabre (GPU + suficiente RAM).
MODO = "dev"
FRACCION_DEV = 0.03   # 3% de ~52M filas ~= 1.5M filas, suficiente para EDA/prototipo

assert os.path.exists(RUTA_DATASET), f"No se encontro {RUTA_DATASET}."
print("Dataset:", RUTA_DATASET, f"({os.path.getsize(RUTA_DATASET)/1e9:.2f} GB)")
print(f"Modo: {MODO}" + (f" (muestra {FRACCION_DEV*100:.0f}%)" if MODO == "dev" else " (dataset completo)"))


## 1. Auto-detección de esquema

In [ ]:
lf = pl.scan_parquet(RUTA_DATASET)
esquema = lf.collect_schema()
columnas = list(esquema.names())
print(f"Columnas detectadas ({len(columnas)}):", columnas)

COLUMNAS_ESPERADAS = ["delta_t", "brightness", "bright_t31", "frp", "scan", "track",
                       "mes", "hora", "es_noche", "latitude", "longitude", "es_falsa_alarma"]
faltantes = [c for c in COLUMNAS_ESPERADAS if c not in columnas]
extra = [c for c in columnas if c not in COLUMNAS_ESPERADAS]
if faltantes:
    print(f"AVISO: faltan columnas esperadas: {faltantes}")
if extra:
    print(f"Columnas adicionales no esperadas: {extra}")

TARGET = TARGET_ESPERADO if TARGET_ESPERADO in columnas else columnas[-1]
print(f"\nTarget: '{TARGET}'")


## 2. Número de registros y columnas

In [ ]:
t0 = time.perf_counter()
n_filas_total = lf.select(pl.len()).collect().item()
print(f"Registros totales en el archivo: {n_filas_total:,}  |  Columnas: {len(columnas)}  ({time.perf_counter()-t0:.1f}s)")

# --- Aplicar submuestreo si estamos en modo dev -------------------------
# gather_every reduce el trabajo DESDE la lectura, no despues: es la clave para que
# esto corra rapido en una laptop en vez de leer 1.83GB completos cada vez.
if MODO == "dev":
    paso = max(1, int(1 / FRACCION_DEV))
    lf = lf.gather_every(paso)
    n_filas = lf.select(pl.len()).collect().item()
    print(f"Modo dev: usando 1 de cada {paso} filas -> {n_filas:,} filas de trabajo")
else:
    n_filas = n_filas_total


## 3. Valores faltantes por columna

In [ ]:
nulos = lf.select([pl.col(c).null_count().alias(c) for c in columnas]).collect(engine="streaming")
resumen_nulos = pd.DataFrame({
    "columna": columnas,
    "tipo": [str(esquema[c]) for c in columnas],
    "nulos": [nulos[c][0] for c in columnas],
})
# .astype("float64") por seguridad ante overflow de uint32
resumen_nulos["pct_nulos"] = 100 * resumen_nulos["nulos"].astype("float64") / n_filas
resumen_nulos = resumen_nulos.sort_values("pct_nulos", ascending=False)
resumen_nulos.to_csv(os.path.join(DIR_TABLAS, "01_resumen_nulos.csv"), index=False)
resumen_nulos


## 4. Distribución de clases (`es_falsa_alarma`)

In [ ]:
balance = lf.group_by(TARGET).agg(pl.len().alias("n")).sort(TARGET).collect(engine="streaming").to_pandas()
# .astype("float64") evita overflow de uint32 en datasets grandes
balance["pct"] = 100 * balance["n"].astype("float64") / balance["n"].sum()
balance.to_csv(os.path.join(DIR_TABLAS, "02_balance_clases.csv"), index=False)
print(balance)

fig = px.pie(balance, names=TARGET, values="n", title=f"Distribucion de clases: {TARGET}")
fig.show()

plt.figure(figsize=(4,4))
plt.pie(balance["n"], labels=balance[TARGET], autopct="%1.1f%%", colors=["#2980b9","#c0392b"])
plt.title(f"Distribucion de clases: {TARGET}"); plt.tight_layout()
plt.savefig(os.path.join(DIR_FIGURAS, "01_balance_clases.png"), dpi=150); plt.close()


## 5. Correlaciones

Siempre sobre una muestra acotada (~200k filas), independientemente del `MODO`, para que este paso nunca sea el cuello de botella de memoria.

In [ ]:
cols_numericas = [c for c, t in esquema.items() if t.is_numeric() and c != TARGET]
paso_corr = max(1, n_filas // 200_000)
muestra = lf.select(cols_numericas + [TARGET]).gather_every(paso_corr).collect(engine="streaming").to_pandas()

corr = muestra[cols_numericas].corr()
fig = px.imshow(corr, text_auto=".2f", color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
                 title="Matriz de correlacion (muestra)")
fig.show()

plt.figure(figsize=(7,6))
im = plt.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
plt.xticks(range(len(cols_numericas)), cols_numericas, rotation=45, ha="right")
plt.yticks(range(len(cols_numericas)), cols_numericas)
plt.colorbar(im); plt.title("Matriz de correlacion"); plt.tight_layout()
plt.savefig(os.path.join(DIR_FIGURAS, "02_correlaciones.png"), dpi=150); plt.close()


## 6. Split train/val/test (70/15/15, estratificado)

Dos claves para que esto no se coma la RAM:
1. Se seleccionan solo las columnas necesarias **antes** de materializar (`lf.select(...)`), nunca las 12+ columnas completas si no hacen falta.
2. Se castea a `float32` en vez de `float64` (mitad de memoria, sin perdida relevante de precision para estos modelos).

En `MODO="dev"` esto procesa la muestra (rapido, para probar el pipeline). En `MODO="full"`
(Kabré) procesa el dataset completo — ahí sí vas a necesitar la RAM/GPU del clúster.


In [ ]:
from sklearn.model_selection import train_test_split

df = (
    lf.select(cols_numericas + [TARGET])
    .with_columns([pl.col(c).cast(pl.Float32) for c in cols_numericas])
    .collect(engine="streaming")
    .to_pandas()
)
print("Dataset cargado en memoria:", df.shape, f"(~{df.memory_usage(deep=True).sum()/1e6:.0f} MB)")

X = df[cols_numericas]
y = df[TARGET]

X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=SEMILLA)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=SEMILLA)

for nombre, (XX, yy) in {"train": (X_train, y_train), "val": (X_val, y_val), "test": (X_test, y_test)}.items():
    out = XX.copy()
    out[TARGET] = yy.values
    out.to_parquet(os.path.join(DIR_DATOS, f"{nombre}.parquet"), index=False)
    print(f"{nombre:<6} n={len(yy):>12,}  pct_positivos={100*yy.mean():.2f}%  -> guardado")


## 7. Resumen

- Esquema validado y comparado contra lo esperado de `preparar_dataset_modelo.py`.
- Nulos, balance de clases y correlaciones documentados y exportados como figuras/tablas.
- Splits `train/val/test.parquet` listos para los notebooks 02-04 — **recuerda que si corriste
  en `MODO="dev"` estos splits son de la muestra, no del dataset completo**. Antes de la
  entrega final, vuelve a correr este notebook en Kabré con `MODO="full"`.

## 8. Cómo correr la versión completa en Kabré

No necesitas convertir nada a mano: Kabré ya tiene JupyterHub, así que puedes abrir este
mismo `.ipynb` ahí y solo cambiar `MODO = "full"`. Pasos:

```bash
# 1) Subir la carpeta modelado/ completa (P mayuscula en scp)
scp -P 22022 -r modelado ulead-17@kabre.cenat.ac.cr:~/

# 2) Copiar el dataset grande a /data, NO a /home (cuota de 10GB en home)
#    (hacelo directo en el clúster, por ssh)
ssh -p 22022 ulead-17@kabre.cenat.ac.cr
mkdir -p /data/$USER
# copia o descarga tu dataset_modelo.parquet ahi, y ajusta RUTA_DATASET si hace falta

# 3) Entrar a JupyterHub de Kabre desde el navegador (pedile la URL a Esteban,
#    el que administra el repo/acceso), abrir modelado/notebooks/01_exploracion_dataset.ipynb
#    ahi, y correr con MODO="full"
```

Si prefieren no usar JupyterHub, tambien se puede correr como script batch (igual que el
benchmark de Esteban en `entrega2_benchmark/`):

```bash
jupyter nbconvert --to script 01_exploracion_dataset.ipynb
# edita la copia .py: cambia MODO = "full"
sbatch submit_kabre.slurm   # reutilizando/adaptando el que ya armo Esteban
squeue -u $USER
tail -f bench_<jobid>.out
```
